## परिचय 

इस पाठ में कवर किया जाएगा: 
- फंक्शन कॉलिंग क्या है और इसके उपयोग के मामले क्या हैं 
- OpenAI का उपयोग करके फंक्शन कॉल कैसे बनाएं 
- एक एप्लिकेशन में फंक्शन कॉल को कैसे इंटीग्रेट करें 

## सीखने के लक्ष्य 

इस पाठ को पूरा करने के बाद आप जानेंगे कैसे करना है और समझेंगे: 

- फंक्शन कॉलिंग का उद्देश्य 
- OpenAI सेवा का उपयोग करके फंक्शन कॉल सेटअप करना 
- अपने एप्लिकेशन के उपयोग के मामले के लिए प्रभावकारी फंक्शन कॉल डिज़ाइन करना 


## फ़ंक्शन कॉल्स को समझना

इस पाठ के लिए, हम अपने शिक्षा स्टार्टअप के लिए एक फीचर बनाना चाहते हैं जो उपयोगकर्ताओं को एक चैटबॉट का उपयोग करके तकनीकी पाठ्यक्रम खोजने की अनुमति देता है। हम ऐसे पाठ्यक्रमों की सिफारिश करेंगे जो उनकी कौशल स्तर, वर्तमान भूमिका और रुचि की तकनीक के अनुसार उपयुक्त हों।

इसे पूरा करने के लिए हम निम्न सम्मिलित करेंगे:
 - उपयोगकर्ता के लिए चैट अनुभव बनाने के लिए `OpenAI`
 - उपयोगकर्ता की अनुरोध के आधार पर पाठ्यक्रम खोजने में मदद के लिए `Microsoft Learn Catalog API`
 - उपयोगकर्ता की क्वेरी लेकर API अनुरोध करने के लिए `Function Calling`

शुरू करने के लिए, आइए देखें कि हम सबसे पहले फ़ंक्शन कॉलिंग क्यों उपयोग करना चाहेंगे:


### फ़ंक्शन कॉलिंग क्यों  

यदि आपने इस कोर्स में कोई अन्य लेसन पूरा किया है, तो आप शायद बड़े भाषा मॉडल (LLMs) का उपयोग करने की ताकत को समझते हैं। उम्मीद है कि आप उनकी कुछ सीमाएँ भी देख पा रहे होंगे।  

फ़ंक्शन कॉलिंग ओपनएआई सेवा की एक सुविधा है जो निम्नलिखित चुनौतियों को संबोधित करने के लिए डिज़ाइन की गई है:  

असंगत प्रतिक्रिया स्वरूपण:  
- फ़ंक्शन कॉलिंग से पहले, एक बड़े भाषा मॉडल से प्रतिक्रियाएँ असंरचित और असंगत होती थीं। डेवलपर्स को प्रत्येक आउटपुट वेरिएशन को संभालने के लिए जटिल सत्यापन कोड लिखना पड़ता था।  

बाहरी डेटा के साथ सीमित एकीकरण:  
- इस सुविधा से पहले, चैट संदर्भ में किसी एप्लिकेशन के अन्य हिस्सों से डेटा शामिल करना मुश्किल था।  

प्रतिक्रिया स्वरूपों को मानकीकृत करके और बाहरी डेटा के साथ सहज एकीकरण सक्षम करके, फ़ंक्शन कॉलिंग विकास को सरल बनाता है और अतिरिक्त सत्यापन तर्क की आवश्यकता को कम करता है।  

उपयोगकर्ता ऐसे उत्तर प्राप्त नहीं कर पाते थे जैसे कि "स्टॉकहोम में वर्तमान मौसम कैसा है?"। ऐसा इसलिए था क्योंकि मॉडल डेटा प्रशिक्षण के समय तक सीमित थे।  

आइए नीचे एक उदाहरण देखें जो इस समस्या को दर्शाता है:  

मान लीजिए हम विद्यार्थी डेटा का एक डेटाबेस बनाना चाहते हैं ताकि हम उन्हें सही कोर्स सुझा सकें। नीचे हमारे पास दो विद्यार्थियों के विवरण हैं जो उनके डेटा में बहुत समान हैं।  


In [ ]:
student_1_description="Emily Johnson is a sophomore majoring in computer science at Duke University. She has a 3.7 GPA. Emily is an active member of the university's Chess Club and Debate Team. She hopes to pursue a career in software engineering after graduating."
 
student_2_description = "Michael Lee is a sophomore majoring in computer science at Stanford University. He has a 3.8 GPA. Michael is known for his programming skills and is an active member of the university's Robotics Club. He hopes to pursue a career in artificial intelligence after finishing his studies."


हम इसे डेटा पार्स करने के लिए एक LLM को भेजना चाहते हैं। इसे बाद में हमारे एप्लिकेशन में उपयोग किया जा सकता है ताकि इसे API को भेजा जा सके या डेटाबेस में संग्रहित किया जा सके।

चलिए दो एकसमान प्रॉम्प्ट बनाते हैं जिनमें हम LLM को निर्देश देते हैं कि हम किस जानकारी में रुचि रखते हैं:


हम इसे एक LLM को भेजना चाहते हैं ताकि हमारे उत्पाद के लिए महत्वपूर्ण भागों को पार्स किया जा सके। इसलिए हम LLM को निर्देश देने के लिए दो समान प्रॉम्प्ट बना सकते हैं: 


In [ ]:
prompt1 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_1_description}
'''


prompt2 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_2_description}
'''


इन दो प्रॉम्प्ट्स को बनाने के बाद, हम उन्हें `client.responses.create` का उपयोग करके LLM को भेजेंगे। हम प्रॉम्प्ट को `input` वेरिएबल में स्टोर करते हैं और भूमिका को `user` असाइन करते हैं। यह एक उपयोगकर्ता द्वारा चैटबॉट के लिए लिखा गया संदेश दर्शाने के लिए है। 



In [ ]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

client = OpenAI()

deployment="gpt-5-mini"


: 

अब हम दोनों अनुरोधों को LLM को भेज सकते हैं और जो प्रतिक्रिया हमें प्राप्त होती है उसका निरीक्षण कर सकते हैं। 


In [ ]:
openai_response1 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt1}],
 store=False,
)
openai_response1.output_text 


In [ ]:
openai_response2 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt2}],
 store=False,
)
openai_response2.output_text


In [ ]:
# Loading the response as a JSON object
json_response1 = json.loads(openai_response1.output_text)
json_response1


In [ ]:
# Loading the response as a JSON object
json_response2 = json.loads(openai_response2.output_text)
json_response2



भले ही प्रम्प्ट एक जैसे हों और विवरण समान हों, फिर भी हमें `Grades` प्रॉपर्टी के विभिन्न प्रारूप मिल सकते हैं।

यदि आप ऊपर दिए गए सेल को कई बार चलाते हैं, तो प्रारूप `3.7` या `3.7 GPA` हो सकता है।

इसका कारण यह है कि LLM लिखित प्रम्प्ट के रूप में असंरचित डेटा लेता है और असंरचित डेटा ही वापस करता है। हमें एक संरचित प्रारूप की आवश्यकता है ताकि हम जान सकें कि इस डेटा को स्टोर या उपयोग करते समय क्या अपेक्षा करनी है।

फंक्शनल कॉलिंग का उपयोग करके, हम यह सुनिश्चित कर सकते हैं कि हमें संरचित डेटा वापस मिले। फंक्शन कॉलिंग का उपयोग करते समय, LLM वास्तविक में कोई फ़ंक्शन कॉल या रन नहीं करता। इसके बजाय, हम LLM के लिए उसके उत्तरों का अनुसरण करने के लिए एक संरचना बनाते हैं। फिर हम उन संरचित उत्तरों का उपयोग यह जानने के लिए करते हैं कि हमारे अनुप्रयोगों में कौन सा फ़ंक्शन चलाना है।
 


![फ़ंक्शन कॉलिंग फ्लो डायग्राम](../../../../translated_images/hi/Function-Flow.083875364af4f4bb.webp)


हम फिर फंक्शन से जो वापस मिलता है उसे ले सकते हैं और इसे LLM को वापस भेज सकते हैं। LLM तब प्राकृतिक भाषा का उपयोग करके उपयोगकर्ता के प्रश्न का उत्तर देगा। 


### फ़ंक्शन कॉल्स के उपयोग के मामले  

**बाहरी टूल्स को कॉल करना**  
चैटबॉट्स उपयोगकर्ताओं के प्रश्नों के उत्तर देने में उत्कृष्ट होते हैं। फ़ंक्शन कॉलिंग का उपयोग करके, चैटबॉट्स उपयोगकर्ताओं के संदेशों का उपयोग कुछ कार्यों को पूरा करने के लिए कर सकते हैं। उदाहरण के लिए, एक छात्र चैटबॉट से कह सकता है "मेरे शिक्षकों को ईमेल भेजें कि मुझे इस विषय में अधिक सहायता चाहिए"। यह `send_email(to: string, body: string)` फ़ंक्शन कॉल कर सकता है।  


**एपीआई या डेटाबेस क्वेरी बनाना**  
उपयोगकर्ता प्राकृतिक भाषा का उपयोग करके जानकारी खोज सकते हैं जिसे एक स्वरूपित क्वेरी या एपीआई अनुरोध में परिवर्तित किया जाता है। इसका एक उदाहरण एक शिक्षक हो सकता है जो पूछता है "वे छात्र कौन हैं जिन्होंने अंतिम असाइनमेंट पूरा किया" जो `get_completed(student_name: string, assignment: int, current_status: string)` नामक फ़ंक्शन को कॉल कर सकता है।  


**संरचित डेटा बनाना**  
उपयोगकर्ता किसी टेक्स्ट ब्लॉक या CSV को लेकर उससे महत्वपूर्ण जानकारी निकालने के लिए LLM का उपयोग कर सकते हैं। उदाहरण के लिए, एक छात्र शांति समझौतों के बारे में विकिपीडिया लेख को एआई फ्लैश कार्ड बनाने के लिए परिवर्तित कर सकता है। यह `get_important_facts(agreement_name: string, date_signed: string, parties_involved: list)` नामक फ़ंक्शन का उपयोग करके किया जा सकता है।  


## 2. अपना पहला फ़ंक्शन कॉल बनाना 

फ़ंक्शन कॉल बनाने की प्रक्रिया में 3 मुख्य चरण शामिल हैं: 
1. अपनी फ़ंक्शनों की सूची और एक उपयोगकर्ता संदेश के साथ चैट कॉम्पलीशंस API को कॉल करना 
2. मॉडल की प्रतिक्रिया पढ़ना ताकि कोई कार्रवाई की जा सके जैसे कि किसी फ़ंक्शन या API कॉल को निष्पादित करना 
3. अपनी फ़ंक्शन से प्राप्त उत्तर के साथ चैट कॉम्पलीशंस API को एक और कॉल करना, ताकि उस जानकारी का उपयोग करके उपयोगकर्ता को प्रतिक्रिया तैयार की जा सके। 


![एक फ़ंक्शन कॉल का प्रवाह](../../../../translated_images/hi/LLM-Flow.3285ed8caf4796d7.webp)


### फ़ंक्शन कॉल के तत्व 

#### उपयोगकर्ता इनपुट 

पहला चरण एक उपयोगकर्ता संदेश बनाना है। इसे एक टेक्स्ट इनपुट का मान लेकर गतिशील रूप से असाइन किया जा सकता है या आप यहाँ एक मान असाइन कर सकते हैं। यदि यह आपका पहली बार है जब आप Chat Completions API के साथ काम कर रहे हैं, तो हमें संदेश की `role` और `content` को परिभाषित करना होगा। 

`role` या तो `system` (नियम बनाना), `assistant` (मॉडल) या `user` (अंतिम उपयोगकर्ता) हो सकता है। फ़ंक्शन कॉलिंग के लिए, हम इसे `user` और एक उदाहरण प्रश्न के रूप में असाइन करेंगे। 


In [ ]:
messages= [ {"role": "user", "content": "Find me a good course for a beginner student to learn Azure."} ]

### फ़ंक्शन बनाना। 

अब हम एक फ़ंक्शन और उस फ़ंक्शन के पैरामीटर को परिभाषित करेंगे। हम यहाँ केवल एक फ़ंक्शन का उपयोग करेंगे जिसे `search_courses` कहा जाता है, लेकिन आप कई फ़ंक्शन बना सकते हैं।

**महत्वपूर्ण** : फ़ंक्शन सिस्टम संदेश में LLM को शामिल किए जाते हैं और उपलब्ध टोकन की मात्रा में शामिल होंगे जो आपके पास उपलब्ध हैं। 


In [ ]:
# The Responses API uses a flat tool format: name/description/parameters at the top level
functions = [
   {
      "type":"function",
      "name":"search_courses",
      "description":"Retrieves courses from the search index based on the parameters provided",
      "parameters":{
         "type":"object",
         "properties":{
            "role":{
               "type":"string",
               "description":"The role of the learner (i.e. developer, data scientist, student, etc.)"
            },
            "product":{
               "type":"string",
               "description":"The product that the lesson is covering (i.e. Azure, Power BI, etc.)"
            },
            "level":{
               "type":"string",
               "description":"The level of experience the learner has prior to taking the course (i.e. beginner, intermediate, advanced)"
            }
         },
         "required":[
            "role"
         ]
      }
   }
]


**परिभाषाएँ** 

फ़ंक्शन परिभाषा संरचना में कई स्तर होते हैं, जिनमें से प्रत्येक के अपने गुण होते हैं। यहाँ नेस्टेड संरचना का विवरण है:

**शीर्ष स्तरीय फ़ंक्शन गुण:**

`name` - उस फ़ंक्शन का नाम जिसे हम कॉल करना चाहते हैं। 

`description` - यह इस बात का वर्णन है कि फ़ंक्शन कैसे काम करता है। यहाँ विशेष रूप से स्पष्ट और सटीक होना महत्वपूर्ण है 

`parameters` - उन मानों और प्रारूपों की एक सूची जो आप चाहते हैं कि मॉडल अपनी प्रतिक्रिया में उत्पन्न करे 

**पैरामीटर ऑब्जेक्ट गुण:**

`type` - पैरामीटर ऑब्जेक्ट का डेटा टाइप (आमतौर पर "object")

`properties` - विशिष्ट मानों की सूची जो मॉडल अपनी प्रतिक्रिया के लिए उपयोग करेगा 

**व्यक्तिगत पैरामीटर गुण:**

`name` - यह प्रॉपर्टी कुंजी द्वारा निहित रूप से परिभाषित होता है (जैसे "role", "product", "level")

`type` - इस विशिष्ट पैरामीटर का डेटा टाइप (जैसे "string", "number", "boolean") 

`description` - विशिष्ट पैरामीटर का विवरण 

**वैकल्पिक गुण:**

`required` - एक सूची जो यह दर्शाती है कि फ़ंक्शन कॉल को पूर्ण करने के लिए कौन-कौन से पैरामीटर आवश्यक हैं 


### फ़ंक्शन कॉल बनाना 
एक फ़ंक्शन को परिभाषित करने के बाद, अब हमें इसे Chat Completion API कॉल में शामिल करना होगा। हम ऐसा `functions` को अनुरोध में जोड़कर करते हैं। इस मामले में `functions=functions` है। 

एक विकल्प यह भी है कि `function_call` को `auto` पर सेट किया जाए। इसका मतलब है कि हम यह निर्णय LLM पर छोड़ देंगे कि उपयोगकर्ता संदेश के आधार पर कौन सा फ़ंक्शन कॉल किया जाना चाहिए बजाय इसके कि हम स्वयं इसे नियुक्त करें।


In [ ]:
response = client.responses.create(model=deployment, 
                                        input=messages,
                                        tools=functions, 
                                        tool_choice="auto",
                                        store=False) 

print(response.output)


अब हम इस प्रतिक्रिया को देखें और देखें कि यह कैसे प्रारूपित है: 

{
  "role": "assistant",
  "function_call": {
    "name": "search_courses",
    "arguments": "{\n  \"role\": \"student\",\n  \"product\": \"Azure\",\n  \"level\": \"beginner\"\n}"
  }
}

आप देख सकते हैं कि फ़ंक्शन का नाम कॉल किया गया है और उपयोगकर्ता संदेश से, LLM तर्कों के अनुसार डेटा खोजने में सक्षम था। 


## 3. फ़ंक्शन कॉल को एक एप्लिकेशन में एकीकृत करना। 


जब हमने LLM से फ़ॉर्मेट किए गए प्रतिक्रिया का परीक्षण कर लिया है, तो अब हम इसे एक एप्लिकेशन में एकीकृत कर सकते हैं। 

### प्रवाह का प्रबंधन करना 

इसे हमारे एप्लिकेशन में एकीकृत करने के लिए, चलिए निम्नलिखित कदम उठाते हैं: 

सबसे पहले, चलिए OpenAI सेवाओं को कॉल करते हैं और संदेश को `response_message` नामक एक वेरिएबल में स्टोर करते हैं। 


In [ ]:
# Extract the function call items from the response output
tool_calls = [item for item in response.output if item.type == "function_call"]


अब हम वह फ़ंक्शन परिभाषित करेंगे जो कोर्स की सूची प्राप्त करने के लिए Microsoft Learn API को कॉल करेगा: 


In [ ]:
import requests

def search_courses(role, product, level):
    url = "https://learn.microsoft.com/api/catalog/"
    params = {
        "role": role,
        "product": product,
        "level": level
    }
    response = requests.get(url, params=params)
    modules = response.json()["modules"]
    results = []
    for module in modules[:5]:
        title = module["title"]
        url = module["url"]
        results.append({"title": title, "url": url})
    return str(results)



एक सर्वश्रेष्ठ अभ्यास के रूप में, हम देखेंगे कि क्या मॉडल कोई फ़ंक्शन कॉल करना चाहता है। उसके बाद, हम उपलब्ध फ़ंक्शनों में से एक बनाएंगे और इसे कॉल किए जा रहे फ़ंक्शन से मेल करेंगे।
फिर हम फ़ंक्शन के तर्कों को लेंगे और उन्हें LLM के तर्कों से मैप करेंगे।

अंत में, हम फ़ंक्शन कॉल संदेश और उन मानों को जोड़ेंगे जो `search_courses` संदेश द्वारा लौटाए गए थे। इससे LLM के पास उपयोगकर्ता को प्राकृतिक भाषा में प्रतिक्रिया देने के लिए सभी जानकारी होती है।
प्रतिक्रिया देने के लिए।


In [ ]:
# Check if the model wants to call a function
if tool_calls:
    tool_call = tool_calls[0]
    print("Recommended Function call:")
    print(tool_call.name)
    print()

    # Call the function. 
    function_name = tool_call.name

    available_functions = {
            "search_courses": search_courses,
    }
    function_to_call = available_functions[function_name] 

    function_args = json.loads(tool_call.arguments)
    function_response = function_to_call(**function_args)

    print("Output of function call:")
    print(function_response)
    print(type(function_response))


    # Add the model's function call item(s) and our function result to the conversation.
    # The Responses API represents tool results as `function_call_output` items.
    messages.extend(response.output)  # adding the model's function_call item(s)
    messages.append( # adding function response to messages
        {
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": function_response,
        }
    )


अब हम LLM को अपडेट किया गया संदेश भेजेंगे ताकि हम API JSON स्वरूपित उत्तर के बजाय एक प्राकृतिक भाषा प्रतिक्रिया प्राप्त कर सकें। 


In [ ]:
print("Messages in next request:")
print(messages)
print()

second_response = client.responses.create(
    input=messages,
    model=deployment,
    tools=functions,
    tool_choice="auto",
    store=False,
        )  # get a new response from the model where it can see the function response


print(second_response.output_text)


## कोड चुनौती 

शानदार काम! OpenAI फ़ंक्शन कॉलिंग सीखना जारी रखने के लिए आप यह बना सकते हैं: https://learn.microsoft.com/training/support/catalog-api-developer-reference?WT.mc_id=academic-105485-koreyst 
 - फ़ंक्शन के अधिक पैरामीटर जो शिक्षार्थियों को और कोर्स खोजने में मदद कर सकते हैं। आप उपलब्ध API पैरामीटर यहां पा सकते हैं: 
 - एक और फ़ंक्शन कॉल बनाएं जो शिक्षार्थी से उनकी मातृभाषा जैसी अधिक जानकारी ले 
 - त्रुटि प्रबंधन बनाएं जब फ़ंक्शन कॉल और/या API कॉल किसी उपयुक्त कोर्स को वापस न करे 


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
इस दस्तावेज़ का अनुवाद AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) का उपयोग करके किया गया है। जबकि हम सटीकता के लिए प्रयास करते हैं, कृपया ध्यान दें कि स्वचालित अनुवादों में त्रुटियाँ या अशुद्धियाँ हो सकती हैं। मूल दस्तावेज़ अपनी मूल भाषा में ही प्रामाणिक स्रोत माना जाना चाहिए। महत्वपूर्ण जानकारी के लिए, पेशेवर मानव अनुवाद की सिफारिश की जाती है। इस अनुवाद के उपयोग से उत्पन्न किसी भी गलतफहमी या गलत व्याख्या के लिए हम उत्तरदायी नहीं हैं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
